In [5]:
import os
import cv2
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# FUNCȚIA DE PREPROCESARE (O Poză -> O Destinație)
# ==========================================
def process_single_image(args):
    img_path, dest_path = args
    
    # Dacă poza a fost deja procesată (în caz de reluare script), dăm skip
    if os.path.exists(dest_path): 
        return True
    
    try:
        img = cv2.imread(img_path)
        if img is None: 
            return False
        
        # 1. Bounding Box & Tight Crop (Rapid)
        small_img = cv2.resize(img, (512, 512))
        gray_small = cv2.cvtColor(small_img, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray_small, 10, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if contours:
            c = max(contours, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(c)
            scale_y, scale_x = img.shape[0] / 512, img.shape[1] / 512
            X, Y, W, H = int(x * scale_x), int(y * scale_y), int(w * scale_x), int(h * scale_y)
            
            center_x, center_y = X + W//2, Y + H//2
            radius = max(W, H) // 2
            start_x, end_x = max(0, center_x - radius), min(img.shape[1], center_x + radius)
            start_y, end_y = max(0, center_y - radius), min(img.shape[0], center_y + radius)
            
            img_cropped = img[start_y:end_y, start_x:end_x]
        else:
            img_cropped = img
            
        # 2. Resize la 512x512
        size = 512
        img_resized = cv2.resize(img_cropped, (size, size))
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
        
        # 3. Ben Graham
        blur = cv2.GaussianBlur(gray, (0, 0), size / 30.0)
        ben_graham = cv2.addWeighted(gray, 4, blur, -4, 128)
        
        # Salvăm direct la destinația stabilită
        cv2.imwrite(dest_path, cv2.cvtColor(ben_graham, cv2.COLOR_GRAY2BGR))
        return True
    
    except Exception as e:
        print(f"\n❌ Eroare la {os.path.basename(img_path)}: {e}")
        return False


# ==========================================
# EXECUTIA PRINCIPALA 
# ==========================================
if __name__ == '__main__':
    print("=== 🟢 SCRIPT SPLIT NATURAL 70/15/15 (TEST 50-50) ===")

    INPUT_CSV = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\originals\EyePACS\all_labels.csv'
    INPUT_DIR = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\originals\EyePACS\Images'
    OUT_DIR_BINAR = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_binar'
    
    # 1. Creare Structură Foldere
    for split in ['train', 'val', 'test']:
        for cls_name in ['sanatos', 'bolnav']:
            os.makedirs(os.path.join(OUT_DIR_BINAR, split, cls_name), exist_ok=True)

    # 2. Încărcare și verificare
    print("\nCitim CSV-ul original...")
    df = pd.read_csv(INPUT_CSV)
    df['file_path'] = df['image'].apply(lambda x: os.path.join(INPUT_DIR, str(x) + '.png')) 
    df['is_diseased'] = (df['level'] > 0).astype(int)

    df['exists'] = df['file_path'].apply(os.path.exists)
    poze_gasite = df['exists'].sum()
    print(f"Am găsit {poze_gasite} imagini valide pe disc.")
    if poze_gasite == 0:
        raise ValueError("Nu am găsit nicio imagine! Verifică extensia pozelor (.jpeg vs .jpg).")
    df = df[df['exists']].drop(columns=['exists']).reset_index(drop=True)

    # ==========================================
    # 3. LOGICA NOUĂ DE ÎMPĂRȚIRE (FĂRĂ DUPLICATE)
    # ==========================================
    print("\nCalculăm matematica pentru Test Set (15% din total, 50/50 sănătos/bolnav)...")
    
    total_images = len(df)
    test_size_total = int(total_images * 0.15) # 15% din total
    test_size_per_class = test_size_total // 2 # Jumătate sănătoși, jumătate bolnavi
    
    # Separăm DataFrame-ul pe clase
    df_0 = df[df['is_diseased'] == 0]
    df_1 = df[df['is_diseased'] == 1]
    
    # Extragem FIX numărul necesar pentru setul de test din fiecare clasă
    test_0 = df_0.sample(n=test_size_per_class, random_state=42)
    test_1 = df_1.sample(n=test_size_per_class, random_state=42)
    df_test = pd.concat([test_0, test_1]).sample(frac=1, random_state=42).reset_index(drop=True)
    df_test['split'] = 'test'
    
    # Rămășițele (scoatem imaginile de test din cele originale)
    df_0_rem = df_0.drop(test_0.index)
    df_1_rem = df_1.drop(test_1.index)
    df_rem = pd.concat([df_0_rem, df_1_rem]).reset_index(drop=True)
    
    # Acum împărțim restul de poze în Train (70%) și Val (15%)
    # Cum 70 + 15 = 85, procentul de Val din rest este 15/85
    val_fraction = 15.0 / 85.0
    
    df_train, df_val = train_test_split(df_rem, test_size=val_fraction, stratify=df_rem['is_diseased'], random_state=42)
    df_train = df_train.copy(); df_train['split'] = 'train'
    df_val = df_val.copy();   df_val['split'] = 'val'

    # Combinăm totul pentru procesare
    df_final = pd.concat([df_train, df_val, df_test]).reset_index(drop=True)

    print("\nSTATISTICI FINALE (FĂRĂ DUPLICATE):")
    print(f"📊 TRAIN (70%): {len(df_train)} imagini (Distribuție naturală)")
    print(f"📊 VAL   (15%): {len(df_val)} imagini (Distribuție naturală)")
    print(f"📊 TEST  (15%): {len(df_test)} imagini (Fix {len(test_0)} Sănătoși / {len(test_1)} Bolnavi)\n")
    print(f"Total general: {len(df_final)} imagini")

    # ==========================================
    # 4. PROCESAREA EFECTIVĂ
    # ==========================================
    print("\nConstruim rutele de salvare...")
    tasks = []
    for _, row in df_final.iterrows():
        cls_name = 'bolnav' if row['is_diseased'] == 1 else 'sanatos'
        dest_path = os.path.join(OUT_DIR_BINAR, row['split'], cls_name, os.path.basename(row['file_path']))
        tasks.append((row['file_path'], dest_path))

    print(f"🚀 Pornim procesarea pe {len(tasks)} fire de execuție...")
    cv2.setNumThreads(0)
    
    max_threads = min(12, (os.cpu_count() or 4) * 2) 
    
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        list(tqdm(executor.map(process_single_image, tasks), total=len(tasks)))
    
    print(f"\n✅ GATA! Datasetul este curat, nedistrusionat și salvat în:\n{OUT_DIR_BINAR}")

=== 🟢 SCRIPT SPLIT NATURAL 70/15/15 (TEST 50-50) ===

Citim CSV-ul original...
Am găsit 88700 imagini valide pe disc.

Calculăm matematica pentru Test Set (15% din total, 50/50 sănătos/bolnav)...

STATISTICI FINALE (FĂRĂ DUPLICATE):
📊 TRAIN (70%): 62090 imagini (Distribuție naturală)
📊 VAL   (15%): 13306 imagini (Distribuție naturală)
📊 TEST  (15%): 13304 imagini (Fix 6652 Sănătoși / 6652 Bolnavi)

Total general: 88700 imagini

Construim rutele de salvare...
🚀 Pornim procesarea pe 88700 fire de execuție...


  0%|          | 0/88700 [00:00<?, ?it/s]


❌ Eroare la 32253_right.png: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


❌ Eroare la 43457_left.png: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


❌ Eroare la 35762_left.png: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\resize.cpp:4208: error: (-215:Assertion failed) !ssize.empty() in function 'cv::resize'


✅ GATA! Datasetul este curat, nedistrusionat și salvat în:
B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_binar


In [1]:
import os
import cv2
import random
import numpy as np
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# FUNCȚIA DE AUGMENTARE PENTRU O SINGURĂ IMAGINE
# ==========================================
def process_augmentation(args):
    src_path, dest_path = args
    
    if os.path.exists(dest_path):
        return True
        
    try:
        img = cv2.imread(src_path)
        if img is None:
            return False
            
        # 1. Flip Orizontal Aleatoriu (50% șanse)
        if random.random() > 0.5:
            img = cv2.flip(img, 1)
            
        # 2. Flip Vertical Aleatoriu (50% șanse)
        if random.random() > 0.5:
            img = cv2.flip(img, 0)
            
        # 3. Rotație Aleatorie (între -15 și +15 grade)
        # Rotim în jurul centrului, iar zonele goale (marginile) rămân negre (borderValue=0)
        angle = random.uniform(-15.0, 15.0)
        h, w = img.shape[:2]
        M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
        img = cv2.warpAffine(img, M, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
        
        # Salvăm noua imagine augmentată
        cv2.imwrite(dest_path, img)
        return True
        
    except Exception as e:
        print(f"❌ Eroare la augmentarea pozei {src_path}: {e}")
        return False

# ==========================================
# EXECUȚIA PRINCIPALĂ
# ==========================================
if __name__ == '__main__':
    print("=== 🧪 SCRIPT AUGMENTARE OFFLINE (BALANSARE TRAIN SET) ===")

    # Calea către folderul tău de TRAIN
    TRAIN_DIR = r'B:\Projects\Disertatie\Diabetic-Retinopathy-Classifier\datasets\processed_by_me\eyepacs\eyepacs_binar\train'
    
    DIR_SANATOS = os.path.join(TRAIN_DIR, 'sanatos')
    DIR_BOLNAV = os.path.join(TRAIN_DIR, 'bolnav')

    # 1. Numărăm imaginile actuale
    poze_sanatosi = [f for f in os.listdir(DIR_SANATOS) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    poze_bolnavi = [f for f in os.listdir(DIR_BOLNAV) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    num_sanatosi = len(poze_sanatosi)
    num_bolnavi = len(poze_bolnavi)

    print(f"📊 Situație actuală în TRAIN:")
    print(f"   - Sănătoși : {num_sanatosi} poze")
    print(f"   - Bolnavi  : {num_bolnavi} poze")

    # 2. Calculăm diferența
    deficit = num_sanatosi - num_bolnavi

    if deficit <= 0:
        print("\n✅ Clasa 'bolnav' este deja egală sau mai mare! Nu este nevoie de augmentare.")
        exit()

    print(f"\n⚠️ Deficit de {deficit} imagini în clasa 'bolnav'.")
    print("Generăm imagini noi (rotite/întoarse) pentru a echilibra clasele...")

    # 3. Creăm lista de task-uri
    tasks = []
    
    # Adăugăm exact atâtea task-uri câte imagini ne lipsesc, alegând poze 'bolnav' la întâmplare
    for i in range(deficit):
        # Alegem o imagine bază la întâmplare din cele existente
        poza_sursa = random.choice(poze_bolnavi)
        src_path = os.path.join(DIR_BOLNAV, poza_sursa)
        
        # Generăm un nume nou: numeoriginal_aug_1.jpeg
        nume_baza, ext = os.path.splitext(poza_sursa)
        nume_nou = f"{nume_baza}_aug_{i}{ext}"
        dest_path = os.path.join(DIR_BOLNAV, nume_nou)
        
        tasks.append((src_path, dest_path))

    # 4. Procesare Multithreading pe Windows
    print(f"🚀 Pornim generarea a {len(tasks)} imagini augmentate pe mai multe fire de execuție...")
    cv2.setNumThreads(0)
    
    max_threads = min(12, (os.cpu_count() or 4) * 2)
    
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        list(tqdm(executor.map(process_augmentation, tasks), total=len(tasks)))

    # 5. Verificare Finală
    num_bolnavi_final = len([f for f in os.listdir(DIR_BOLNAV) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
    print("\n=== 🎯 REZULTAT FINAL TRAIN SET ===")
    print(f"📊 Sănătoși: {num_sanatosi}")
    print(f"📊 Bolnavi : {num_bolnavi_final}")
    print("✅ Echilibrare completă! Acum poți reîncepe antrenamentul.")

=== 🧪 SCRIPT AUGMENTARE OFFLINE (BALANSARE TRAIN SET) ===
📊 Situație actuală în TRAIN:
   - Sănătoși : 48332 poze
   - Bolnavi  : 13756 poze

⚠️ Deficit de 34576 imagini în clasa 'bolnav'.
Generăm imagini noi (rotite/întoarse) pentru a echilibra clasele...
🚀 Pornim generarea a 34576 imagini augmentate pe mai multe fire de execuție...


  0%|          | 0/34576 [00:00<?, ?it/s]


=== 🎯 REZULTAT FINAL TRAIN SET ===
📊 Sănătoși: 48332
📊 Bolnavi : 48332
✅ Echilibrare completă! Acum poți reîncepe antrenamentul.
